In [0]:
df = spark.sql("select * from databricksreddy.silver.orders")

In [0]:
df.display()

In [0]:
df_dim_customers = spark.sql('''select DimCustomerKey, customer_id as dim_customer_id from databricksreddy.gold.dimcustomers''')

df_dim_products = spark.sql('''select product_id as DimProductKey, product_id as dim_product_id from databricksreddy.gold.dimproductsnew''')


In [0]:
df_dim_products.display()

In [0]:
df_dim_customers.display()

In [0]:
df_fact = df.join(df_dim_customers, df.customer_id == df_dim_customers.dim_customer_id, "left").join(df_dim_products, df.product_id == df_dim_products.dim_product_id, "left")

df_fact_new = df_fact.drop('dim_customer_id', 'dim_product_id','customer_id','product_id')

In [0]:
df_fact_new.write.mode("overwrite").saveAsTable("databricksreddy.gold.facttable")

In [0]:
df_fact_new.display()

In [0]:
%sql
select * from databricksreddy.gold.facttable

##Upsert on Fact Table

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists('databricksreddy.gold.facttable'):
    DeltaTable.forName(spark,'databricksreddy.gold.facttable').alias('trg').merge(df_fact_new.alias('src'), 'trg.order_id = src.order_id and trg.DimCustomerKey = src.DimCustomerKey and trg.DimProductKey = src.DimProductKey').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
  df_fact_new.write.mode("overwrite").option("path","abfss://gold@datalakevishnu1.dfs.core.windows.net/FactTable")